# Ablation: Random vs Smart Initialization
So sanh optimizer hinh hoc tren tap test [quangne/CGL-Text2Geo](https://huggingface.co/datasets/quangne/CGL-Text2Geo) (381 samples), moi DSL toi uu **1 lan/mode**, seed co dinh.

**Dau ra:**
- `report.csv` - loss, epochs, runtime, coid degenerate *tham khao*, duong dan anh
- `gallery.html` - anh random | smart dat canh nhau de ban **duyet bang mat**
- Bang LaTeX tong hop o cell cuoi

**Cach dung:** chay lan luot cell 1 den 5. Cell 4 chay ~70 phut; neu Colab ngat thi chay lai cell 4 (co resume).


In [ ]:
#@title 1. Clone repo + ap dung patch
import base64, pathlib

REPO = "https://github.com/johnpham4/GeoSystem.git"

if not pathlib.Path('/content/GeoSystem').exists():
    !git clone --depth 1 https://github.com/johnpham4/GeoSystem.git /content/GeoSystem

PATCH_B64 = "..."
%cd /content/GeoSystem
pathlib.Path("colab_bundle.patch").write_bytes(base64.b64decode(PATCH_B64))
!git apply colab_bundle.patch && echo "PATCH OK"


In [ ]:
#@title 2. Cai dependencies
!pip -q install loguru datasets matplotlib


In [ ]:
#@title 3. Smoke test (~4 phut) - co render anh xem thu gallery
!python -u profiling/ablation_initializer.py --repo quangne/CGL-Text2Geo --split test --field answer --max-samples 6 --seeds 1 --epochs 1000 --workers 2 --render-dir profiling/render --output profiling/colab_smoke.json

from IPython.display import HTML
HTML(filename="profiling/render/gallery.html")


In [ ]:
#@title 4. CHAY FULL - moi DSL toi uu 1 lan/mode, kem anh (~70 phut Colab free)
#@markdown Co --resume: neu Colab ngat, chay lai cell nay - tiep tuc tu cho dung.
!python -u profiling/ablation_initializer.py --repo quangne/CGL-Text2Geo --split test --field answer --seeds 1 --epochs 1000 --workers 2 --render-dir profiling/render --output profiling/ablation_cgl_test.json


In [ ]:
#@title 5. Ket qua tong hop + bang LaTeX + gallery
import json
from IPython.display import HTML

with open('profiling/ablation_cgl_test.json', encoding='utf-8') as f:
    payload = json.load(f)

print("Elapsed:", payload["elapsed_seconds"], "s")
for mode, s in payload['summaries'].items():
    print()
    print(f'[{mode}]')
    print(f'  Success rate   : {s["success_rate_pct"]}% ({s["successes"]}/{s["total_runs"]})')
    print(f'  Avg epochs     : {s["avg_epochs"]} +/- {s["avg_epochs_std"]}')
    print(f'  Epochs to tau  : {s["avg_epochs_to_tau"]} +/- {s["avg_epochs_to_tau_std"]} (reached: {s["runs_reached_tau"]})')
    print(f'  Avg solve time : {s["avg_time_s"]} s +/- {s["avg_time_s_std"]}')
    print(f'  Final loss     : {s["final_loss_mean"]} +/- {s["final_loss_std"]}')
    print(f'  Degenerate     : {s["degenerate_pct"]}%')

print()
print(payload.get('latex', ''))

HTML(filename='profiling/render/gallery.html')
# from google.colab import drive
# drive.mount('/content/drive')
# !cp profiling/ablation_cgl_test*.json /content/drive/MyDrive/
# !zip -r /content/drive/MyDrive/render.zip profiling/render/
